# House Prices Competition - Modeling Notebook

**Objective:** Build and evaluate regression models to predict `SalePrice` (log-transformed)

**Metrics:** RMSE (on log scale) → RMSLE on original scale

**Approach:** Ridge baseline → XGBoost → LightGBM → Ensemble

**Validation:** 5-fold KFold with target encoding inside CV (no leakage)

## 1. Setup & Imports

In [90]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modeling
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.preprocessing import LabelEncoder

# Advanced models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Hyperparameter tuning
import optuna
from optuna.samplers import TPESampler

# Misc
import joblib
from datetime import datetime

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Processed Data

In [91]:
# Load datasets
X_train = pd.read_csv("../data/X_train_processed.csv")
y_train = pd.read_csv("../data/y_train_processed.csv").values.ravel()  # Log-transformed SalePrice
X_test = pd.read_csv("../data/X_test_processed.csv")

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny_train stats - mean: {y_train.mean():.4f}, std: {y_train.std():.4f}")

# %%
# Quick sanity check - no missing values should remain
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

# Verify outlier flags are present (if added in preprocessing)
outlier_flags = [col for col in X_train.columns if 'IsLarge' in col]
if outlier_flags:
    print(f"\nOutlier flag columns present: {outlier_flags}")
else:
    print("\nNo outlier flag columns found (will proceed without them)")

X_train shape: (1460, 201)
y_train shape: (1460,)
X_test shape: (1459, 201)

y_train stats - mean: 12.0241, std: 0.3993
Missing values in X_train: 0
Missing values in X_test: 0

Outlier flag columns present: ['IsLargeGrLivArea', 'IsLargeTotalBsmtSF', 'IsLarge1stFlrSF']


## 3. Identify Columns for Target Encoding

In [92]:
# Columns that need target encoding (high cardinality)
target_encode_cols = ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']

# Verify they exist
for col in target_encode_cols:
    if col in X_train.columns:
        print(f"✓ {col} found - {X_train[col].nunique()} unique values")
    else:
        print(f"✗ {col} NOT found - check preprocessing")

✓ MSSubClass found - 15 unique values
✓ Neighborhood found - 25 unique values
✓ Exterior1st found - 15 unique values
✓ Exterior2nd found - 16 unique values


## 4. Simple Target Encoding Function (No Leakage)

For each fold: compute means on training fold, apply to validation fold

In [93]:
def apply_target_encoding(X_train_fold, y_train_fold, X_val_fold, columns, global_mean, smoothing=20):
    """
    Apply target encoding to validation fold based on training fold means.
    
    Parameters:
    - X_train_fold, X_val_fold: DataFrames
    - y_train_fold: numpy array or pandas Series
    - columns: list of column names to encode
    - global_mean: float (mean of target on full training data)
    - smoothing: float (smoothing factor)
    
    Returns:
        X_train_enc, X_val_enc (with encoded columns added)
    """
    X_train_enc = X_train_fold.copy()
    X_val_enc = X_val_fold.copy()
    
    # Convert y to pandas Series if it's a numpy array
    if isinstance(y_train_fold, np.ndarray):
        y_train_fold = pd.Series(y_train_fold, index=X_train_fold.index)
    
    for col in columns:
        # Compute group means on training fold
        group_means = y_train_fold.groupby(X_train_fold[col]).agg(['mean', 'count'])
        
        # Apply smoothing: (mean * count + global_mean * smoothing) / (count + smoothing)
        smoothed = (group_means['mean'] * group_means['count'] + global_mean * smoothing) / (group_means['count'] + smoothing)
        
        # Create encoded column name
        encoded_col = f'{col}_target_enc'
        
        # Add to validation fold
        X_val_enc[encoded_col] = X_val_fold[col].map(smoothed).fillna(global_mean)
        
        # Add to training fold (using same mapping for consistency)
        X_train_enc[encoded_col] = X_train_fold[col].map(smoothed).fillna(global_mean)
    
    return X_train_enc, X_val_enc

## 5. Cross-Validation Function with Target Encoding

In [94]:
def evaluate_with_target_encoding(model, model_name, X, y, encode_cols, n_folds=5):
    """
    Evaluate model using KFold with target encoding inside the loop.
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []
    predictions = np.zeros(len(y))
    global_mean = y.mean()
    
    print(f"\n{model_name} - {n_folds}-fold CV with target encoding for: {encode_cols}")
    print("-" * 70)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]
        
        # Apply target encoding
        X_train_enc, X_val_enc = apply_target_encoding(
            X_train_fold, y_train_fold, X_val_fold, 
            encode_cols, global_mean, smoothing=20
        )
        
        # Drop original encoded columns (now replaced by encoded versions)
        X_train_enc = X_train_enc.drop(columns=encode_cols)
        X_val_enc = X_val_enc.drop(columns=encode_cols)
        
        # Train and predict
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_train_enc, y_train_fold)
        y_pred = model_clone.predict(X_val_enc)
        predictions[val_idx] = y_pred
        
        score = rmse(y_val_fold, y_pred)
        cv_scores.append(score)
        print(f"  Fold {fold+1}: RMSE = {score:.5f}")
    
    mean_score = np.mean(cv_scores)
    std_score = np.std(cv_scores)
    print(f"\n  Mean RMSE: {mean_score:.5f} (+/- {std_score:.5f})")
    
    return predictions, cv_scores

## 6. Baseline: Ridge Regression

In [95]:
# Quick alpha tuning for Ridge (using 3-fold CV for speed)
alphas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]
ridge_cv_scores = []

print("="*70)
print("Tuning Ridge Regression Alpha")
print("="*70)

for alpha in alphas:
    ridge = Ridge(alpha=alpha, random_state=RANDOM_STATE)
    _, scores = evaluate_with_target_encoding(
        ridge, f"Ridge (alpha={alpha})", 
        X_train, y_train, target_encode_cols, n_folds=3
    )
    ridge_cv_scores.append(np.mean(scores))

best_alpha = alphas[np.argmin(ridge_cv_scores)]
print(f"\n{'='*70}")
print(f"🏆 Best alpha: {best_alpha} (CV RMSE: {min(ridge_cv_scores):.5f})")
print(f"{'='*70}")

# %%
# Final Ridge with best alpha (5-fold CV)
ridge_final = Ridge(alpha=best_alpha, random_state=RANDOM_STATE)
ridge_preds, ridge_scores = evaluate_with_target_encoding(
    ridge_final, f"Ridge (alpha={best_alpha}) - FINAL", 
    X_train, y_train, target_encode_cols, n_folds=5
)

Tuning Ridge Regression Alpha

Ridge (alpha=0.01) - 3-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
----------------------------------------------------------------------
  Fold 1: RMSE = 0.16441
  Fold 2: RMSE = 0.17698
  Fold 3: RMSE = 0.15836

  Mean RMSE: 0.16658 (+/- 0.00775)

Ridge (alpha=0.1) - 3-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
----------------------------------------------------------------------
  Fold 1: RMSE = 0.16090
  Fold 2: RMSE = 0.17773
  Fold 3: RMSE = 0.14354

  Mean RMSE: 0.16072 (+/- 0.01396)

Ridge (alpha=0.5) - 3-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
----------------------------------------------------------------------
  Fold 1: RMSE = 0.14591
  Fold 2: RMSE = 0.17591
  Fold 3: RMSE = 0.12700

  Mean RMSE: 0.14960 (+/- 0.02014)

Ridge (alpha=1.0) - 3-fold CV with target encoding for: ['MSSubClass

## 7. XGBoost with GPU

In [96]:
# XGBoost baseline (5-fold CV)
xgb_base = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    device='cuda',  # Use GPU
    verbosity=0
)

print("\n" + "="*70)
print("XGBoost Baseline")
print("="*70)
xgb_preds, xgb_scores = evaluate_with_target_encoding(
    xgb_base, "XGBoost (baseline)", 
    X_train, y_train, target_encode_cols, n_folds=5
)


XGBoost Baseline

XGBoost (baseline) - 5-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
----------------------------------------------------------------------
  Fold 1: RMSE = 0.13825
  Fold 2: RMSE = 0.11708
  Fold 3: RMSE = 0.15435
  Fold 4: RMSE = 0.13176
  Fold 5: RMSE = 0.11152

  Mean RMSE: 0.13059 (+/- 0.01530)


### XGBoost Hyperparameter Tuning

In [97]:
def objective_xgb(trial, X, y, encode_cols, cv_folds):
    """Optuna objective function for XGBoost"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1200, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5, log=True),
        'random_state': RANDOM_STATE,
        'device': 'cuda',
        'verbosity': 0
    }
    
    scores = []
    global_mean = y.mean()
    
    for train_idx, val_idx in cv_folds:
        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]
        
        # Target encoding
        X_train_enc, X_val_enc = apply_target_encoding(
            X_train_fold, y_train_fold, X_val_fold, 
            encode_cols, global_mean, smoothing=20
        )
        X_train_enc = X_train_enc.drop(columns=encode_cols)
        X_val_enc = X_val_enc.drop(columns=encode_cols)
        
        model = XGBRegressor(**params)
        model.fit(X_train_enc, y_train_fold)
        y_pred = model.predict(X_val_enc)
        scores.append(rmse(y_val_fold, y_pred))
    
    return np.mean(scores)

In [98]:
# Create fold indices for tuning (3-fold for speed)
kf_tune = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(kf_tune.split(X_train))

print("\n" + "="*70)
print("XGBoost Hyperparameter Tuning (30 trials)")
print("="*70)

study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(
    lambda trial: objective_xgb(trial, X_train, y_train, target_encode_cols, fold_indices), 
    n_trials=30, 
    show_progress_bar=True
)

print(f"\n{'='*70}")
print(f"🏆 Best XGBoost CV RMSE: {study_xgb.best_value:.5f}")
print("Best parameters:")
for key, value in study_xgb.best_params.items():
    print(f"  {key}: {value}")
print(f"{'='*70}")

[I 2026-05-25 03:19:44,223] A new study created in memory with name: no-name-14a63bad-83e1-44cf-a11d-241d25752883



XGBoost Hyperparameter Tuning (30 trials)


Best trial: 0. Best value: 0.141224:   3%|▎         | 1/30 [00:05<02:48,  5.81s/it]

[I 2026-05-25 03:19:50,036] Trial 0 finished with value: 0.14122437680065714 and parameters: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.07259248719561363, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 0.0005407712220288125, 'reg_lambda': 0.00018747059221802545}. Best is trial 0 with value: 0.14122437680065714.


Best trial: 1. Best value: 0.134217:   7%|▋         | 2/30 [00:15<03:47,  8.14s/it]

...

[I 2026-05-25 03:22:46,706] Trial 27 finished with value: 0.13019695360228625 and parameters: {'n_estimators': 1200, 'max_depth': 5, 'learning_rate': 0.014217376303869942, 'subsample': 0.7707864727761808, 'colsample_bytree': 0.9035458971227509, 'reg_alpha': 0.14093928584940232, 'reg_lambda': 0.09532526789655701}. Best is trial 20 with value: 0.12824536374241924.


Best trial: 20. Best value: 0.128245:  97%|█████████▋| 29/30 [03:09<00:07,  7.19s/it]

[I 2026-05-25 03:22:53,487] Trial 28 finished with value: 0.130324958405258 and parameters: {'n_estimators': 1000, 'max_depth': 4, 'learning_rate': 0.0612552268177223, 'subsample': 0.7056231418860678, 'colsample_bytree': 0.8508123832767526, 'reg_alpha': 0.4731787370286036, 'reg_lambda': 0.010689431795409661}. Best is trial 20 with value: 0.12824536374241924.


Best trial: 20. Best value: 0.128245: 100%|██████████| 30/30 [03:13<00:00,  6.45s/it]

[I 2026-05-25 03:22:57,593] Trial 29 finished with value: 0.13682989663936496 and parameters: {'n_estimators': 700, 'max_depth': 3, 'learning_rate': 0.08662029017206618, 'subsample': 0.8171256311395682, 'colsample_bytree': 0.9688926188663773, 'reg_alpha': 0.0145153248201911, 'reg_lambda': 0.0003154525925427716}. Best is trial 20 with value: 0.12824536374241924.

🏆 Best XGBoost CV RMSE: 0.12825
Best parameters:
  n_estimators: 900
  max_depth: 4
  learning_rate: 0.018103950419565393
  subsample: 0.7056281465849481
  colsample_bytree: 0.873643739405751
  reg_alpha: 0.6483919038070519
  reg_lambda: 0.00013747658844454524


In [99]:
# Final XGBoost with best parameters (5-fold CV)
xgb_final = XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, device='cuda', verbosity=0)
print("\n" + "="*70)
print("XGBoost Optimized - FINAL EVALUATION")
print("="*70)
xgb_preds, xgb_scores = evaluate_with_target_encoding(
    xgb_final, "XGBoost (optimized)", 
    X_train, y_train, target_encode_cols, n_folds=5
)


XGBoost Optimized - FINAL EVALUATION

XGBoost (optimized) - 5-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
----------------------------------------------------------------------
  Fold 1: RMSE = 0.13368
  Fold 2: RMSE = 0.11121
  Fold 3: RMSE = 0.14413
  Fold 4: RMSE = 0.12794
  Fold 5: RMSE = 0.10987

  Mean RMSE: 0.12537 (+/- 0.01318)


## 8. LightGBM with GPU

In [100]:
# LightGBM baseline (5-fold CV)
lgb_base = LGBMRegressor(
    n_estimators=500,
    num_leaves=31,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    device='gpu',
    verbosity=-1
)

print("\n" + "="*70)
print("LightGBM Baseline")
print("="*70)
lgb_preds, lgb_scores = evaluate_with_target_encoding(
    lgb_base, "LightGBM (baseline)", 
    X_train, y_train, target_encode_cols, n_folds=5
)



LightGBM Baseline

LightGBM (baseline) - 5-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
----------------------------------------------------------------------
  Fold 1: RMSE = 0.14205
  Fold 2: RMSE = 0.11598
  Fold 3: RMSE = 0.15439
  Fold 4: RMSE = 0.13196
  Fold 5: RMSE = 0.11553

  Mean RMSE: 0.13198 (+/- 0.01503)


### LightGBM Hyperparameter Tuning

In [101]:
def objective_lgb(trial, X, y, encode_cols, cv_folds):
    """Optuna objective function for LightGBM"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1200, step=100),
        'num_leaves': trial.suggest_int('num_leaves', 15, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5, log=True),
        'random_state': RANDOM_STATE,
        'device': 'gpu',
        'verbosity': -1
    }
    
    scores = []
    global_mean = y.mean()
    
    for train_idx, val_idx in cv_folds:
        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]
        
        X_train_enc, X_val_enc = apply_target_encoding(
            X_train_fold, y_train_fold, X_val_fold, 
            encode_cols, global_mean, smoothing=20
        )
        X_train_enc = X_train_enc.drop(columns=encode_cols)
        X_val_enc = X_val_enc.drop(columns=encode_cols)
        
        model = LGBMRegressor(**params)
        model.fit(X_train_enc, y_train_fold)
        y_pred = model.predict(X_val_enc)
        scores.append(rmse(y_val_fold, y_pred))
    
    return np.mean(scores)

In [103]:
print("\n" + "="*70)
print("LightGBM Hyperparameter Tuning (30 trials)")
print("="*70)

study_lgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))
study_lgb.optimize(
    lambda trial: objective_lgb(trial, X_train, y_train, target_encode_cols, fold_indices), 
    n_trials=30, 
    show_progress_bar=True
)

print(f"\n{'='*70}")
print(f"🏆 Best LightGBM CV RMSE: {study_lgb.best_value:.5f}")
print("Best parameters:")
for key, value in study_lgb.best_params.items():
    print(f"  {key}: {value}")
print(f"{'='*70}")

[I 2026-05-25 03:29:29,857] A new study created in memory with name: no-name-66ed6ff8-823f-4a26-a8c5-39a9e1dab6a1



LightGBM Hyperparameter Tuning (30 trials)


Best trial: 0. Best value: 0.133807:   3%|▎         | 1/30 [00:07<03:45,  7.77s/it]

[I 2026-05-25 03:29:37,622] Trial 0 finished with value: 0.13380685615750795 and parameters: {'n_estimators': 600, 'num_leaves': 96, 'max_depth': 10, 'learning_rate': 0.050591436432963696, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00018747059221802545, 'reg_lambda': 1.1752647960576206}. Best is trial 0 with value: 0.13380685615750795.


Best trial: 0. Best value: 0.133807:   7%|▋         | 2/30 [00:11<02:23,  5.13s/it]

...

[I 2026-05-25 03:32:45,145] Trial 27 finished with value: 0.13263522687581214 and parameters: {'n_estimators': 1000, 'num_leaves': 81, 'max_depth': 8, 'learning_rate': 0.04269169272665067, 'subsample': 0.939617640383007, 'colsample_bytree': 0.6909255010328326, 'reg_alpha': 0.3350053128220192, 'reg_lambda': 0.18276680158186473}. Best is trial 3 with value: 0.13018883438277523.


Best trial: 28. Best value: 0.129826:  97%|█████████▋| 29/30 [03:19<00:06,  6.87s/it]

[I 2026-05-25 03:32:48,903] Trial 28 finished with value: 0.1298262777749769 and parameters: {'n_estimators': 700, 'num_leaves': 52, 'max_depth': 4, 'learning_rate': 0.03144456246517308, 'subsample': 0.8432992844993579, 'colsample_bytree': 0.649526944806543, 'reg_alpha': 0.0025728516316320953, 'reg_lambda': 0.0002978568269057051}. Best is trial 28 with value: 0.1298262777749769.


Best trial: 28. Best value: 0.129826: 100%|██████████| 30/30 [03:22<00:00,  6.74s/it]

[I 2026-05-25 03:32:52,039] Trial 29 finished with value: 0.1308139275293578 and parameters: {'n_estimators': 600, 'num_leaves': 38, 'max_depth': 4, 'learning_rate': 0.058071092997062666, 'subsample': 0.6838935969487516, 'colsample_bytree': 0.650773441009233, 'reg_alpha': 0.011744450140718544, 'reg_lambda': 0.00023389488209048657}. Best is trial 28 with value: 0.1298262777749769.

🏆 Best LightGBM CV RMSE: 0.12983
Best parameters:
  n_estimators: 700
  num_leaves: 52
  max_depth: 4
  learning_rate: 0.03144456246517308
  subsample: 0.8432992844993579
  colsample_bytree: 0.649526944806543
  reg_alpha: 0.0025728516316320953
  reg_lambda: 0.0002978568269057051


In [106]:
# Final LightGBM with best parameters (5-fold CV)
lgb_final = LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, device='gpu', verbosity=-1)
print("\n" + "="*70)
print("LightGBM Optimized - FINAL EVALUATION")
print("="*70)
lgb_preds, lgb_scores = evaluate_with_target_encoding(
    lgb_final, "LightGBM (optimized)", 
    X_train, y_train, target_encode_cols, n_folds=5
)


LightGBM Optimized - FINAL EVALUATION

LightGBM (optimized) - 5-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
----------------------------------------------------------------------
  Fold 1: RMSE = 0.13685
  Fold 2: RMSE = 0.11092
  Fold 3: RMSE = 0.15033
  Fold 4: RMSE = 0.12901
  Fold 5: RMSE = 0.10893

  Mean RMSE: 0.12721 (+/- 0.01569)


## 9. Model Comparison & Ensemble

In [107]:
# Compile results
results_df = pd.DataFrame({
    'Model': ['Ridge', 'XGBoost', 'LightGBM'],
    'CV RMSE': [np.mean(ridge_scores), np.mean(xgb_scores), np.mean(lgb_scores)],
    'Std': [np.std(ridge_scores), np.std(xgb_scores), np.std(lgb_scores)]
})

print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)



MODEL COMPARISON SUMMARY
   Model  CV RMSE      Std
   Ridge 0.141680 0.036194
 XGBoost 0.125366 0.013177
LightGBM 0.127205 0.015686


In [ ]:
# Weighted ensemble (inverse RMSE weights) # This seems to be a simple weighting technique
weights = {
    'ridge': 1 / np.mean(ridge_scores),
    'xgb': 1 / np.mean(xgb_scores),
    'lgb': 1 / np.mean(lgb_scores)
}
total_weight = sum(weights.values())
weights = {k: v/total_weight for k, v in weights.items()}

print("\nEnsemble weights:")
for model, weight in weights.items():
    print(f"  {model}: {weight:.4f}")

# Calculate ensemble CV performance
ensemble_preds = (
    weights['ridge'] * ridge_preds +
    weights['xgb'] * xgb_preds +
    weights['lgb'] * lgb_preds
)
ensemble_rmse = rmse(y_train, ensemble_preds)

print(f"\n{'='*70}")
print(f"🎯 Ensemble CV RMSE: {ensemble_rmse:.5f}")
print(f"{'='*70}")


Ensemble weights:
  ridge: 0.3083
  xgb: 0.3484
  lgb: 0.3433

🎯 Ensemble CV RMSE: 0.12671


## 10. Train Final Models on Full Dataset

In [109]:
print("\n" + "="*70)
print("Training Final Models on Complete Dataset")
print("="*70)

# Prepare training data with target encoding (using full dataset)
global_mean = y_train.mean()
X_train_encoded = X_train.copy()

for col in target_encode_cols:
    # Compute encodings on FULL training set
    if isinstance(y_train, np.ndarray):
        y_train_series = pd.Series(y_train, index=X_train.index)
    else:
        y_train_series = y_train
    
    group_means = y_train_series.groupby(X_train[col]).agg(['mean', 'count'])
    smoothed = (group_means['mean'] * group_means['count'] + global_mean * 20) / (group_means['count'] + 20)
    encoded_col = f'{col}_target_enc'
    X_train_encoded[encoded_col] = X_train[col].map(smoothed).fillna(global_mean)

# Drop original columns
X_train_final = X_train_encoded.drop(columns=target_encode_cols)

print(f"Final training shape: {X_train_final.shape}")

# Train all models
ridge_full = Ridge(alpha=best_alpha, random_state=RANDOM_STATE)
ridge_full.fit(X_train_final, y_train)
print("✓ Ridge trained")

xgb_full = XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, device='cuda', verbosity=0)
xgb_full.fit(X_train_final, y_train)
print("✓ XGBoost trained")

lgb_full = LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, device='gpu', verbosity=-1)
lgb_full.fit(X_train_final, y_train)
print("✓ LightGBM trained")


Training Final Models on Complete Dataset
Final training shape: (1460, 201)
✓ Ridge trained
✓ XGBoost trained
✓ LightGBM trained


## 11. Generate Test Predictions

In [112]:
print("\n" + "="*70)
print("Generating Test Predictions")
print("="*70)

# Prepare test data with target encoding
X_test_encoded = X_test.copy()

for col in target_encode_cols:
    # Use same encodings computed from training data
    if isinstance(y_train, np.ndarray):
        y_train_series = pd.Series(y_train, index=X_train.index)
    else:
        y_train_series = y_train
    
    group_means = y_train_series.groupby(X_train[col]).agg(['mean', 'count'])
    smoothed = (group_means['mean'] * group_means['count'] + global_mean * 20) / (group_means['count'] + 20)
    encoded_col = f'{col}_target_enc'
    X_test_encoded[encoded_col] = X_test[col].map(smoothed).fillna(global_mean)

# Drop original columns from test
X_test_final = X_test_encoded.drop(columns=target_encode_cols)

# CRITICAL: Ensure test has the SAME columns as training (in same order)
print(f"\nBefore alignment:")
print(f"  Training columns: {X_train_final.shape[1]}")
print(f"  Test columns: {X_test_final.shape[1]}")

# Align test columns with training columns
# This ensures same columns and same order
missing_cols = set(X_train_final.columns) - set(X_test_final.columns)
extra_cols = set(X_test_final.columns) - set(X_train_final.columns)

if missing_cols:
    print(f"\n  Adding missing columns to test: {missing_cols}")
    for col in missing_cols:
        X_test_final[col] = 0

if extra_cols:
    print(f"  Removing extra columns from test: {extra_cols}")
    X_test_final = X_test_final.drop(columns=extra_cols)

# Reorder test columns to match training exactly
X_test_final = X_test_final[X_train_final.columns]

print(f"\nAfter alignment:")
print(f"  Training columns: {X_train_final.shape[1]}")
print(f"  Test columns: {X_test_final.shape[1]}")
print(f"  Columns match: {X_train_final.columns.equals(X_test_final.columns)}")

# Verify no missing values
print(f"\nMissing values in test: {X_test_final.isnull().sum().sum()}")

# Generate predictions
ridge_preds_log = ridge_full.predict(X_test_final)
xgb_preds_log = xgb_full.predict(X_test_final)
lgb_preds_log = lgb_full.predict(X_test_final)

ensemble_preds_log = (
    weights['ridge'] * ridge_preds_log +
    weights['xgb'] * xgb_preds_log +
    weights['lgb'] * lgb_preds_log
)

# Convert back to original scale (inverse of log1p)
ensemble_preds_original = np.expm1(ensemble_preds_log)

print(f"\nPrediction statistics (original scale):")
print(f"  Min: ${ensemble_preds_original.min():,.0f}")
print(f"  Max: ${ensemble_preds_original.max():,.0f}")
print(f"  Mean: ${ensemble_preds_original.mean():,.0f}")
print(f"  Median: ${np.median(ensemble_preds_original):,.0f}")


Generating Test Predictions

Before alignment:
  Training columns: 201
  Test columns: 201

After alignment:
  Training columns: 201
  Test columns: 201
  Columns match: True

Missing values in test: 0

Prediction statistics (original scale):
  Min: $49,325
  Max: $530,295
  Mean: $175,836
  Median: $154,492


## 12. Create Submission File

In [113]:
# Load test IDs
test_original = pd.read_csv("../data/test.csv")

# Create submission DataFrame
submission = pd.DataFrame({
    'Id': test_original['Id'],
    'SalePrice': ensemble_preds_original.round(0)  # Round to whole dollars
})

# Save submission
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"../submissions/submission_{timestamp}.csv"
submission.to_csv(submission_filename, index=False)

print(f"\n{'='*70}")
print(f"✅ SUBMISSION SAVED")
print(f"{'='*70}")
print(f"File: {submission_filename}")
print(f"Shape: {submission.shape}")
print(f"Sample (first 5 rows):")
print(submission.head())
print(f"{'='*70}")



✅ SUBMISSION SAVED
File: ../submissions/submission_20260525_034553.csv
Shape: (1459, 2)
Sample (first 5 rows):
     Id  SalePrice
0  1461   118022.0
1  1462   157087.0
2  1463   181231.0
3  1464   191260.0
4  1465   181957.0


## 13. Save Model Artifacts

In [115]:
# Save all models and metadata
model_artifacts = {
    'ridge': ridge_full,
    'xgb': xgb_full,
    'lgb': lgb_full,
    'ensemble_weights': weights,
    'target_encode_cols': target_encode_cols,
    'cv_results': {
        'ridge_mean_rmse': np.mean(ridge_scores),
        'ridge_std': np.std(ridge_scores),
        'xgb_mean_rmse': np.mean(xgb_scores),
        'xgb_std': np.std(xgb_scores),
        'lgb_mean_rmse': np.mean(lgb_scores),
        'lgb_std': np.std(lgb_scores),
        'ensemble_rmse': ensemble_rmse
    },
    'best_params': {
        'ridge_alpha': best_alpha,
        'xgb': study_xgb.best_params,
        'lgb': study_lgb.best_params
    },
    'global_mean': global_mean,
    'random_state': RANDOM_STATE
}

joblib.dump(model_artifacts, '../models/model_artifacts.pkl')
print(f"\n✅ Model artifacts saved to: ../models/model_artifacts.pkl")



✅ Model artifacts saved to: ../models/model_artifacts.pkl


## 14. Final Summary

In [116]:
print("\n" + "="*70)
print("🏆 MODELING COMPLETE - FINAL SUMMARY")
print("="*70)
print(f"Dataset:")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Test samples: {len(X_test)}")
print(f"  - Features used: {X_train_final.shape[1]}")
print(f"\nTarget Encoding:")
print(f"  - Columns encoded: {target_encode_cols}")
print(f"  - Method: Group means with smoothing (s=20)")
print(f"\nCross-Validation:")
print(f"  - Strategy: {n_folds}-fold KFold (shuffled)")
print(f"  - Target encoding applied inside CV (no leakage)")
print(f"\nModel Performance (CV RMSE on log scale):")
print(f"  - Ridge: {np.mean(ridge_scores):.5f} (+/- {np.std(ridge_scores):.5f})")
print(f"  - XGBoost: {np.mean(xgb_scores):.5f} (+/- {np.std(xgb_scores):.5f})")
print(f"  - LightGBM: {np.mean(lgb_scores):.5f} (+/- {np.std(lgb_scores):.5f})")
print(f"  - Ensemble: {ensemble_rmse:.5f}")
print(f"\nSubmission:")
print(f"  - File: {submission_filename}")
print(f"  - Ready for Kaggle upload")
print("="*70)


🏆 MODELING COMPLETE - FINAL SUMMARY
Dataset:
  - Training samples: 1460
  - Test samples: 1459
  - Features used: 201

Target Encoding:
  - Columns encoded: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
  - Method: Group means with smoothing (s=20)

Cross-Validation:
  - Strategy: 5-fold KFold (shuffled)
  - Target encoding applied inside CV (no leakage)

Model Performance (CV RMSE on log scale):
  - Ridge: 0.14168 (+/- 0.03619)
  - XGBoost: 0.12537 (+/- 0.01318)
  - LightGBM: 0.12721 (+/- 0.01569)
  - Ensemble: 0.12671

Submission:
  - File: ../submissions/submission_20260525_034553.csv
  - Ready for Kaggle upload


In [118]:
# Optional: Plot feature importance from best model (XGBoost)
feature_importance = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': xgb_full.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (XGBoost):")
for i, row in feature_importance.head(20).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")


Top 20 Most Important Features (XGBoost):
  OverallQual: 0.2687
  TotalSF: 0.0995
  TotalQual: 0.0763
  CentralAir_Y: 0.0486
  Neighborhood_target_enc: 0.0317
  IsLargeTotalBsmtSF: 0.0307
  KitchenQual: 0.0284
  IsLargeGrLivArea: 0.0233
  TotalBath: 0.0220
  IsLarge1stFlrSF: 0.0184
  GarageCars: 0.0172
  PoolQC: 0.0168
  PoolArea: 0.0163
  FireplaceQu: 0.0163
  GarageCond: 0.0157
  GarageQual: 0.0131
  RemodAge: 0.0090
  GrLivArea: 0.0088
  QualPerAge: 0.0084
  GarageType_Detchd: 0.0078


## Next Steps (Optional Improvements)

1. **Stacking Ensemble** - Train meta-model on out-of-fold predictions
2. **More tuning trials** - Increase Optuna trials from 30 to 100+
3. **Feature selection** - Remove low-importance features
4. **Advanced imputation** - Try KNN or MICE for missing values
5. **Hyperparameter tuning with full 5-fold CV** (slower but more accurate)

---
**✅ Ready to submit to Kaggle!**